# NFL pregame probability model
Run the cells from top to bottom with **Shift+Enter**, or use **Run All**.
Keep `sample.csv` beside this notebook. A is the listed home team; B is away.
We train on 2023 and evaluate on 2024. Ratings pool the previous four games.
The data exclude ties and early-season games without enough history.

In VS Code, select your `.venv` as the notebook kernel. Install dependencies
in your activated terminal with `python -m pip install -r requirements.txt`.

This notebook follows the simplified script: fixed C, one model, no tuning.
The interaction switch starts off. These are learning experiments; repeated
changes after seeing 2024 results make 2024 part of model development.

## 1. Import the tools

In [13]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss


## 2. Load the games
`A_win` is 1 for a home win and 0 for an away win.

In [14]:
data = pd.read_csv("./data/sample.csv")
print("Number of games:", len(data))
display(data[["game_date", "A_team", "B_team", "A_win"]].head())

Number of games: 416


,game_date,A_team,B_team,A_win
0,2023-10-05,WAS,CHI,0
1,2023-10-08,PIT,BAL,1
2,2023-10-08,DET,CAR,1
3,2023-10-08,ARI,CIN,0
4,2023-10-08,SF,DAL,1


In [17]:
# Download historical game information and moneylines.
url = "https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv"
schedules = pd.read_csv(url)

# Keep only the columns we need.
market_odds = schedules[
    ["game_id", "home_moneyline", "away_moneyline"]
]

# Match the odds to our games using game_id.
data = data.merge(
    market_odds,
    on="game_id",
    how="left",
    validate="one_to_one",
)

# Inspect the games we will test on.
test_games = data[data["season"] == 2024]


# Convert American odds into implied probability.
def implied_probability(odds):
    if pd.isna(odds):
        return float("nan")
    if odds < 0:
        return -odds / (-odds + 100)
    return 100 / (odds + 100)


home_implied = data["home_moneyline"].apply(implied_probability)
away_implied = data["away_moneyline"].apply(implied_probability)

# Remove vig by normalizing the two probabilities to sum to 1.
total_implied = home_implied + away_implied

data["market_home_probability"] = home_implied / total_implied
data["market_away_probability"] = away_implied / total_implied

display(data.loc[
    data["season"] == 2024,
    ["game_date", "A_team", "B_team",
     "home_moneyline", "away_moneyline",
     "market_home_probability", "market_away_probability"],
].head(10))

,game_date,A_team,B_team,home_moneyline,away_moneyline,market_home_probability,market_away_probability
208,2024-10-03,ATL,TB,-142.0,120.0,0.563492,0.436508
209,2024-10-06,SF,ARI,-325.0,260.0,0.733542,0.266458
210,2024-10-06,CIN,BAL,124.0,-148.0,0.427940,0.572060
211,2024-10-06,HOU,BUF,-118.0,-102.0,0.517364,0.482636
212,2024-10-06,CHI,CAR,-198.0,164.0,0.636904,0.363096
213,2024-10-06,WAS,CLE,-155.0,130.0,0.582993,0.417007
214,2024-10-06,PIT,DAL,-135.0,114.0,0.551441,0.448559
215,2024-10-06,LA,GB,130.0,-155.0,0.417007,0.582993
216,2024-10-06,JAX,IND,-185.0,154.0,0.622467,0.377533
217,2024-10-06,DEN,LV,-155.0,130.0,0.582993,0.417007


## 3. Build the features
Start with six home-minus-away rating differences. Higher ratings mean better
performance. Set `USE_INTERACTIONS = True` to also include the three products
discussed earlier, then **rerun this cell and every cell below it**.
Products use raw fractions; scaling happens afterward.

In [18]:
USE_INTERACTIONS = True

ratings = ["Off_pass", "Off_run", "Off_passblock",
           "Def_pass", "Def_run", "Def_passrush"]

X = pd.DataFrame(index=data.index)
for rating in ratings:
    X[rating] = data["A_" + rating] - data["B_" + rating]

if USE_INTERACTIONS:
    X["pass_matchup"] = (
        data["A_Off_pass"] * data["B_Def_pass"]
        - data["B_Off_pass"] * data["A_Def_pass"]
    )
    X["run_matchup"] = (
        data["A_Off_run"] * data["B_Def_run"]
        - data["B_Off_run"] * data["A_Def_run"]
    )
    X["protection_matchup"] = (
        data["A_Off_passblock"] * data["B_Def_passrush"]
        - data["B_Off_passblock"] * data["A_Def_passrush"]
    )

y = data["A_win"]
display(X.head())

,Off_pass,Off_run,Off_passblock,Def_pass,Def_run,Def_passrush,pass_matchup,run_matchup,protection_matchup
0,0.026198,-0.000831,-0.001965,0.040056,-0.066038,0.077922,-0.003059,0.031798,-0.059099
1,-0.146640,0.009509,-0.184646,-0.083593,-0.000195,0.100170,-0.043285,0.005217,-0.120362
2,0.122644,0.090869,-0.016832,-0.003726,0.099199,0.021634,0.068827,0.002602,-0.020893
3,0.079012,-0.017142,0.019880,-0.043273,-0.088612,-0.032840,0.057143,0.032111,0.029963
4,0.061734,0.004393,-0.116198,-0.092568,-0.045357,0.008133,0.087458,0.025580,-0.035398


## 4. Split by season
The model learns from 2023 games and predicts 2024 games.

In [19]:
train_rows = data["season"] == 2023
test_rows = data["season"] == 2024

X_train = X[train_rows]
y_train = y[train_rows]
X_test = X[test_rows]
y_test = y[test_rows]

print("Training games:", len(X_train))
print("Test games:", len(X_test))

Training games: 208
Test games: 208


## 5. Scale the inputs
Learn each feature's mean and standard deviation from training data only.
Use those same values to transform the test data.

In [20]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Train logistic regression
The model learns a coefficient for each feature plus an intercept.
Smaller `C` penalizes large coefficients more strongly. We fix `C=1` here.
`max_iter` limits how many optimization iterations are allowed.

In [21]:
model = LogisticRegression(C=1.0, max_iter=2000)
model.fit(X_train_scaled, y_train)
print("Model trained.")

Model trained.


## 7. Inspect the coefficients
A coefficient is the change in log-odds for a one-standard-deviation increase
in that input, holding other inputs fixed. These are associations, not causal
effects. Correlated features can make individual weights hard to interpret.
Interaction weights describe the product differences, not a simple matchup advantage.

In [22]:
weights = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0],
})
display(weights)
print("Intercept:", model.intercept_[0])

,feature,coefficient
0,Off_pass,0.090167
1,Off_run,0.565184
2,Off_passblock,-0.059657
3,Def_pass,-0.020394
4,Def_run,-0.160116
5,Def_passrush,0.138750
6,pass_matchup,-0.034139
7,run_matchup,-0.375398
8,protection_matchup,-0.118063


Intercept: 0.3438332420240784


## 8. Predict and evaluate
`predict_proba` returns two columns: away win (0), then home win (1).
Log loss measures probability prediction quality; lower is better.
The baseline always predicts the training season's home-win frequency.

In [23]:
probabilities = model.predict_proba(X_test_scaled)[:, 1]

baseline_probability = y_train.mean()
baseline_predictions = [baseline_probability] * len(y_test)

print("Baseline log loss:", round(log_loss(y_test, baseline_predictions), 4))
print("Model log loss:   ", round(log_loss(y_test, probabilities), 4))

Baseline log loss: 0.694
Model log loss:    0.6742


## 9. Look at game predictions
Change `head(10)` to see more games.

In [28]:
results = data.loc[test_rows, [
    "game_date",
    "A_team",
    "B_team",
    "A_win",
    "market_home_probability",
]].copy()

results["model_home_probability"] = probabilities

# Positive = our model favors the home team more than the market.
results["difference_pp"] = 100 * (
    results["model_home_probability"]
    - results["market_home_probability"]
)

display(results.head(50))

,game_date,A_team,B_team,A_win,market_home_probability,model_home_probability,difference_pp
208,2024-10-03,ATL,TB,1,0.563492,0.577486,1.399404
209,2024-10-06,SF,ARI,0,0.733542,0.736366,0.282331
210,2024-10-06,CIN,BAL,0,0.427940,0.416282,-1.165855
211,2024-10-06,HOU,BUF,1,0.517364,0.609152,9.178775
212,2024-10-06,CHI,CAR,1,0.636904,0.705861,6.895673
213,2024-10-06,WAS,CLE,1,0.582993,0.530214,-5.277889
214,2024-10-06,PIT,DAL,0,0.551441,0.563377,1.193617
215,2024-10-06,LA,GB,0,0.417007,0.568795,15.178749
216,2024-10-06,JAX,IND,1,0.622467,0.634624,1.215739
217,2024-10-06,DEN,LV,1,0.582993,0.744026,16.103301


## 10. Explain one game's score
`game_number = 0` means the first test game. Change this to another row number.
Each contribution is the scaled input times its coefficient. Contributions
plus the intercept equal the game's log-odds score. They are not additive
percentage-point changes in win probability.

In [29]:
game_number = 0

display(results.iloc[[game_number]])
contributions = pd.DataFrame({
    "feature": X.columns,
    "scaled_value": X_test_scaled[game_number],
    "coefficient": model.coef_[0],
    "score_contribution": X_test_scaled[game_number] * model.coef_[0],
})
display(contributions)
score = model.intercept_[0] + contributions["score_contribution"].sum()
print("Total log-odds score:", score)

,game_date,A_team,B_team,A_win,market_home_probability,model_home_probability,difference_pp
208,2024-10-03,ATL,TB,1,0.563492,0.577486,1.399404


,feature,scaled_value,coefficient,score_contribution
0,Off_pass,-1.013662,0.090167,-0.091399
1,Off_run,0.926747,0.565184,0.523783
2,Off_passblock,-0.223299,-0.059657,0.013321
3,Def_pass,-0.142347,-0.020394,0.002903
4,Def_run,-0.430404,-0.160116,0.068915
5,Def_passrush,-1.005850,0.138750,-0.139562
6,pass_matchup,-0.592214,-0.034139,0.020217
7,run_matchup,0.894994,-0.375398,-0.335979
8,protection_matchup,0.792542,-0.118063,-0.093570


Total log-odds score: 0.3124620323661883


Data: previously assembled free nflverse play-by-play and PFR pressure
proxies for 2023–2024, retrieved September 29, 2026. Passing/running ratings
use success fractions; defensive versions are one minus opponent success.
Protection is one minus pressures/dropbacks; pass rush is pressures/opponent
dropbacks. No opponent, roster, injury or neutral-venue adjustment is included.